# Phase 7: GenAI Response Automation
**AAI-540 MLOps | CX Cortalyst — Customer Sentiment Analysis & GenAI Response Automation**  
**Author:** Jagadeesh Kumar Sellappan

## Overview
This notebook generates personalized draft email responses for Negative reviews using a JumpStart LLM:

1. **Read** 10 Negative predictions from Batch Transform output (S3)
2. **Join** with original review text
3. **Deploy** FLAN-T5-XL via SageMaker JumpStart (ml.m5.xlarge)
4. **Generate** draft email responses for each review
5. **Display** original review + draft response side by side
6. **Save** to S3
7. **Terminate** endpoint immediately (cost control)

**NOTE:** 
The CI/CD pipeline handles model training and deployment.
 The GenAI response step runs as a downstream process,
 triggered after batch inference completes.
 In production this would be triggered by an S3 event
 when new predictions land in the output bucket.

---
## 1. Environment Setup

In [2]:
!pip uninstall -y sagemaker sagemaker-core sagemaker-train sagemaker-serve sagemaker-mlops sagemaker-studio 

Found existing installation: sagemaker 2.232.2


Uninstalling sagemaker-2.232.2:


  Successfully uninstalled sagemaker-2.232.2


Found existing installation: sagemaker-core 1.0.78
Uninstalling sagemaker-core-1.0.78:
  Successfully uninstalled sagemaker-core-1.0.78
Found existing installation: sagemaker-train 1.11.0


Uninstalling sagemaker-train-1.11.0:
  Successfully uninstalled sagemaker-train-1.11.0


Found existing installation: sagemaker-serve 1.11.0
Uninstalling sagemaker-serve-1.11.0:
  Successfully uninstalled sagemaker-serve-1.11.0


Found existing installation: sagemaker-mlops 1.11.0
Uninstalling sagemaker-mlops-1.11.0:
  Successfully uninstalled sagemaker-mlops-1.11.0


Found existing installation: sagemaker_studio 1.1.19


Uninstalling sagemaker_studio-1.1.19:
  Successfully uninstalled sagemaker_studio-1.1.19


In [4]:
!pip install -q sagemaker==2.232.2 --no-deps
!pip install -q sagemaker-core==1.0.78
!pip install -q boto3 pandas awswrangler

In [1]:
import boto3
import sagemaker
import pandas as pd
import json
import time
import awswrangler as wr
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# ── Configuration ─────────────────────────────────────────────
REGION = 'us-east-1'
BUCKET = 'aai-540-group8-yelp-data-301798465569-us-east-1-an'

PREDICTIONS_PATH = f's3://{BUCKET}/data/predictions/production_predictions.parquet'
PRODUCTION_PATH  = f's3://{BUCKET}/data/splits/production/'
RESPONSES_PATH   = f's3://{BUCKET}/pipeline/draft-responses/'

N_REVIEWS        = 10   # number of negative reviews to process
MODEL_ID         = 'huggingface-text2text-flan-t5-xl'
INSTANCE_TYPE    = 'ml.m5.xlarge'
ENDPOINT_NAME    = f'cx-cortalyst-genai-{int(time.time())}'

boto_session      = boto3.Session(region_name=REGION)
sagemaker_session = sagemaker.Session(boto_session=boto_session)
sm_client         = boto3.client('sagemaker', region_name=REGION)
s3_client         = boto3.client('s3',        region_name=REGION)
role              = sagemaker.get_execution_role(sagemaker_session=sagemaker_session)

print('Environment initialized')
print(f'Model:    {MODEL_ID}')
print(f'Instance: {INSTANCE_TYPE}')
print(f'Reviews:  {N_REVIEWS}')

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml


sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml


Environment initialized
Model:    huggingface-text2text-flan-t5-xl
Instance: ml.m5.xlarge
Reviews:  10


---
## 2. Load 10 Negative Reviews

In [2]:
print('Loading predictions from S3...')

# Load batch transform predictions
df_predictions = pd.read_parquet(PREDICTIONS_PATH)
print(f'Total predictions: {len(df_predictions):,}')
print(f'Negative: {(df_predictions["predicted_label"]==0).sum():,}')
print(f'Positive: {(df_predictions["predicted_label"]==1).sum():,}')

# Get 10 most confident negative predictions
df_negative = df_predictions[
    df_predictions['predicted_label'] == 0
].sort_values('predicted_proba', ascending=True).head(N_REVIEWS).reset_index(drop=True)

print(f'\nSelected {N_REVIEWS} most confident negative reviews')

# Load production split for review text
df_production = pd.read_parquet(PRODUCTION_PATH)

# Join to get review text
df_reviews = df_negative.merge(
    df_production[['review_id', 'text']],
    on='review_id',
    how='left'
).dropna(subset=['text']).reset_index(drop=True)

print(f'Reviews with text: {len(df_reviews)}')
print(f'\nReviews to process:')
print(f'{"#":<4} {"Review ID":<15} {"Neg Confidence":<18} {"Review Preview"}')
print('-' * 80)
for i, row in df_reviews.iterrows():
    preview = str(row['text'])[:50].replace('\n', ' ')
    print(f'{i+1:<4} {row["review_id"][:12]:<15} {row["predicted_proba"]:.1%}{"":12} {preview}...')

Loading predictions from S3...


Total predictions: 5,000
Negative: 1,836
Positive: 3,164

Selected 10 most confident negative reviews


Reviews with text: 10

Reviews to process:
#    Review ID       Neg Confidence     Review Preview
--------------------------------------------------------------------------------
1    I_c5K4J65l5z    0.0%             Worst restaurant experience in my life thus far. I...
2    CqtMBz-Ebjgd    0.0%             I went in this past Friday night . It was really l...
3    ez-00XxIHFzZ    0.0%             Food was horrible and place was disgusting. I orde...
4    p7FFNdF7Anhi    0.0%             I can not believe the service here it might be the...
5    nSRFpxlnYM8q    0.0%             I know times are tough --stopped by the other nigh...
6    dhEZ3CJuLPBD    0.0%             First I should say that I did not hire this guy. W...
7    ipoPvd_emGV2    0.0%             Last week, I watched an employee try to rip off a ...
8    yy_Zw6XlkoSY    0.0%             I don't dislike this Credit Union but I am somewha...
9    qd4jjAexXnwU    0.0%             I found this place to be a total disaster!! Fir

---
## 3. Deploy FLAN-T5-XL via JumpStart

In [3]:
from sagemaker.jumpstart.model import JumpStartModel

print(f'Deploying {MODEL_ID}...')
print(f'Instance: {INSTANCE_TYPE}')
print(f'Endpoint: {ENDPOINT_NAME}')

start = time.time()

model = JumpStartModel(
    model_id=MODEL_ID,
    model_version='*',
    role=role,
    sagemaker_session=sagemaker_session
)

predictor = model.deploy(
    initial_instance_count=1,
    instance_type=INSTANCE_TYPE,
    endpoint_name=ENDPOINT_NAME
)

print(f'\nEndpoint deployed in {time.time()-start:.0f}s')
print(f'Name: {ENDPOINT_NAME}')
print(f'\nRun the TERMINATE cell after generation!')

Deploying huggingface-text2text-flan-t5-xl...
Instance: ml.m5.xlarge
Endpoint: cx-cortalyst-genai-1781578052
⏳ Takes ~5-8 minutes...


Using model 'huggingface-text2text-flan-t5-xl' with wildcard version identifier '*'. You can pin to version '2.2.9' for more stable results. Note that models may have different input/output signatures after a major version upgrade.


-

-

-

...

-

-

-

!


Endpoint deployed in 454s
Name: cx-cortalyst-genai-1781578052

Run the TERMINATE cell after generation!


---
## 4. Generate Draft Email Responses

In [9]:
# ── Replace ONLY these two functions ──────────────────────────

def build_prompt(review_text):
    """Prompt for FLAN-T5 to respond AS the business."""
    truncated = str(review_text)[:400]
    return (
        f"You are a customer service manager responding to a negative "
        f"Yelp review on behalf of your business. Write a short, "
        f"professional apology email to the customer. "
        f"Do not repeat the review. Sign off as 'The Management Team'. "
        f"Keep it under 60 words.\n\n"
        f"Customer review: {truncated}\n\n"
        f"Our response to the customer:"
    )

import re

def clean_response(text):
    """Remove hallucinated names and fix greeting."""
    # Replace any "Dear Mr/Mrs/Ms [Name]" with standard greeting
    text = re.sub(
        r'Dear\s+(Mr\.?|Mrs\.?|Ms\.?|Dr\.?)?\s*\w+,?',
        'Dear Valued Customer,',
        text
    )
    # Remove repeated sentences (FLAN-T5 repeats itself)
    sentences = text.split('. ')
    seen = []
    for s in sentences:
        if s.strip() not in seen:
            seen.append(s.strip())
    return '. '.join(seen).strip()

def generate_response(predictor, prompt):
    payload = {
        "inputs": prompt,
        "parameters": {
            "max_new_tokens": 100,
            "temperature": 0.3,
            "do_sample": False,
        }
    }
    response = predictor.predict(
        payload,
        custom_attributes='accept_eula=true'
    )

    if isinstance(response, list):
        raw = response[0].get('generated_text', '')
    elif isinstance(response, dict):
        raw = response.get('generated_text', '')
    else:
        raw = str(response)

    # Strip prompt echo
    for marker in ["Business owner apology:", "The customer complained:", "Write a short apology"]:
        if marker in raw:
            raw = raw.split(marker)[-1]

    # Clean hallucinated names and repetition
    return clean_response(raw.strip())


# ── Everything below stays exactly the same ───────────────────
print(f'Regenerating responses for {N_REVIEWS} reviews...\n')
results = []
for i, row in df_reviews.iterrows():
    try:
        start   = time.time()
        prompt  = build_prompt(row['text'])
        draft   = generate_response(predictor, prompt)
        elapsed = time.time() - start
        results.append({
            'review_id':        row['review_id'],
            'predicted_proba':  float(row['predicted_proba']),
            'true_label':       int(row['true_label']),
            'original_review':  str(row['text'])[:500],
            'draft_response':   draft,
            'generated_at':     datetime.utcnow().isoformat(),
            'model_id':         MODEL_ID,
            'inference_time_s': round(elapsed, 2)
        })
        print(f'[{i+1:2d}/{N_REVIEWS}] {row["review_id"][:12]}... ({elapsed:.1f}s)')
    except Exception as e:
        print(f'[{i+1:2d}/{N_REVIEWS}] Error: {e}')

print(f'\nGenerated {len(results)}/{N_REVIEWS} responses')

Regenerating responses for 10 reviews...



[ 1/10] I_c5K4J65l5z... (10.8s)


[ 2/10] CqtMBz-Ebjgd... (21.9s)


[ 3/10] ez-00XxIHFzZ... (8.5s)


[ 4/10] p7FFNdF7Anhi... (17.7s)


...

[ 6/10] dhEZ3CJuLPBD... (10.1s)


[ 7/10] ipoPvd_emGV2... (29.8s)


[ 8/10] yy_Zw6XlkoSY... (9.7s)


[ 9/10] qd4jjAexXnwU... (11.8s)


[10/10] Y6tMtXgMcF5r... (9.7s)

Generated 10/10 responses


---
## 5. Display Results — Original Review vs Draft Response

In [10]:
print('='*70)
print('  CX CORTALYST — GENAI DRAFT RESPONSES')
print(f'  Model: {MODEL_ID}  |  Instance: {INSTANCE_TYPE}')
print(f'  Generated: {datetime.utcnow().strftime("%Y-%m-%d %H:%M UTC")}')
print('='*70)

for i, result in enumerate(results):
    print(f'\n── REVIEW {i+1:02d} of {N_REVIEWS} ──────────────────────────────────────')
    print(f'  Review ID:   {result["review_id"]}')
    print(f'  Confidence:  {result["predicted_proba"]:.1%} negative')
    print(f'  Time:        {result["inference_time_s"]}s')

    # Original review — clean and truncated
    review_clean = str(result["original_review"])\
        .replace('\n', ' ')\
        .replace('\r', ' ')\
        .strip()
    print(f'\n  📝 Customer review:')
    print(f'  {review_clean[:300]}{"..." if len(review_clean) > 300 else ""}')

    # Draft response — strip any remaining prompt echo
    draft = result["draft_response"]\
        .replace('\n', ' ')\
        .strip()
    # Remove if model still echoes instruction fragments
    for fragment in [
        "Write a professional", "You are a customer",
        "Customer review:", "Our response to the customer:"
    ]:
        if fragment in draft:
            draft = draft.split(fragment)[-1].strip()

    print(f'\n  ✉️  Draft response (GenAI):')
    print(f'  {draft}')
    print(f'\n  ⏳ Status: PENDING HUMAN AGENT REVIEW')

print(f'\n{"="*70}')
print(f'  Total responses: {len(results)}/10')
print(f'  Avg inference:   {sum(r["inference_time_s"] for r in results)/len(results):.1f}s')
print(f'  Saved to:        s3://{BUCKET}/pipeline/draft-responses/')
print(f'{"="*70}')

  CX CORTALYST — GENAI DRAFT RESPONSES
  Model: huggingface-text2text-flan-t5-xl  |  Instance: ml.m5.xlarge
  Generated: 2026-06-16 03:17 UTC

── REVIEW 01 of 10 ──────────────────────────────────────
  Review ID:   I_c5K4J65l5zhZP8OIdpiA
  Confidence:  0.0% negative
  Time:        10.75s

  📝 Customer review:
  Worst restaurant experience in my life thus far. I was in the area for the big Steeplechase event, and along with 17 of my friends, we decided to go to Barefoot Charlies for the island atmosphere and live music, both of which were great, however the food and the service was THE WORST I have ever exp...

  ✉️  Draft response (GenAI):
  We are so sorry for the experience you had at Barefoot Charlie's. We are working on it. We apologize for the inconvenience.

  ⏳ Status: PENDING HUMAN AGENT REVIEW

── REVIEW 02 of 10 ──────────────────────────────────────
  Review ID:   CqtMBz-EbjgdnFvQFyVvhQ
  Confidence:  0.0% negative
  Time:        21.87s

  📝 Customer review:
  I went in thi

---
## 6. Save to S3

In [11]:
print('Saving draft responses to S3...')

df_responses = pd.DataFrame(results)
timestamp    = datetime.utcnow().strftime('%Y%m%d_%H%M%S')

# Save Parquet
parquet_path = f'{RESPONSES_PATH}responses_{timestamp}.parquet'
wr.s3.to_parquet(df=df_responses, path=parquet_path, index=False)
print(f'Parquet → {parquet_path}')

# Save JSON for human review queue
json_key = f'pipeline/draft-responses/responses_{timestamp}.json'
s3_client.put_object(
    Bucket=BUCKET,
    Key=json_key,
    Body=json.dumps(results, indent=2),
    ContentType='application/json'
)
print(f'JSON    → s3://{BUCKET}/{json_key}')

print(f'''
╔══════════════════════════════════════════════════════════════╗
║   GENAI RESPONSE GENERATION — COMPLETE                      ║
╠══════════════════════════════════════════════════════════════╣
║  Model:     huggingface-text2text-flan-t5-xl                ║
║  Instance:  ml.m5.xlarge                                    ║
║  Reviews:   {len(results):<48} ║
║  Saved to:  s3://{BUCKET[:40]:<40} ║
║  Status:    Pending human agent review                      ║
╚══════════════════════════════════════════════════════════════╝
''')

Saving draft responses to S3...


2026-06-16 03:23:24,070	WARNING services.py:2137 -- WARNING: The object store is using /tmp instead of /dev/shm because /dev/shm has only 408924160 bytes available. This will harm performance! You may be able to free up space by deleting files in /dev/shm. If you are inside a Docker container, you can increase /dev/shm size by passing '--shm-size=0.76gb' to 'docker run' (or add it to the run_options list in a Ray cluster config). Make sure to set this to more than 30% of available RAM.


2026-06-16 03:23:26,281	INFO worker.py:2007 -- Started a local Ray instance.


Parquet → s3://aai-540-group8-yelp-data-301798465569-us-east-1-an/pipeline/draft-responses/responses_20260616_032320.parquet
JSON    → s3://aai-540-group8-yelp-data-301798465569-us-east-1-an/pipeline/draft-responses/responses_20260616_032320.json

╔══════════════════════════════════════════════════════════════╗
║   GENAI RESPONSE GENERATION — COMPLETE                      ║
╠══════════════════════════════════════════════════════════════╣
║  Model:     huggingface-text2text-flan-t5-xl                ║
║  Instance:  ml.m5.xlarge                                    ║
║  Reviews:   10                                               ║
║  Saved to:  s3://aai-540-group8-yelp-data-301798465569-us ║
║  Status:    Pending human agent review                      ║
╚══════════════════════════════════════════════════════════════╝



---
## 7. Terminate Endpoint 

In [12]:
print(f'Terminating endpoint: {ENDPOINT_NAME}')

try:
    predictor.delete_endpoint()
    print(f'Endpoint terminated via predictor')
except Exception as e:
    print(f'Trying boto3 fallback: {e}')
    try:
        sm_client.delete_endpoint(EndpointName=ENDPOINT_NAME)
        print(f'Endpoint terminated via boto3')
    except Exception as e2:
        print(f'Manual deletion needed in AWS Console: {e2}')

# Verify deleted
time.sleep(5)
try:
    sm_client.describe_endpoint(EndpointName=ENDPOINT_NAME)
    print('Endpoint still exists — check AWS Console → Endpoints')
except:
    print('Endpoint confirmed deleted — no ongoing costs')

Terminating endpoint: cx-cortalyst-genai-1781578052


Endpoint terminated via predictor


Endpoint confirmed deleted — no ongoing costs
